In [1]:
# Block 1: Install Dependencies for CAT-DTI
!pip install torch==2.7.1 torch-geometric==2.6.1 scikit-learn matplotlib tqdm pandas requests rdkit-pypi

print("⚠️  IMPORTANT: After running this cell, go to Runtime -> Restart Runtime")
print("   Then run the imports cell below")

print("="*50)
print("STEP 1: Run this cell first")
print("STEP 2: Go to Runtime -> Restart Runtime")
print("STEP 3: Then run the cell below")
print("="*50)

⚠️  IMPORTANT: After running this cell, go to Runtime -> Restart Runtime
   Then run the imports cell below
STEP 1: Run this cell first
STEP 2: Go to Runtime -> Restart Runtime
STEP 3: Then run the cell below


In [ ]:
# Block 2: Imports and Setup for CAT-DTI
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader, Batch
from torch_geometric.nn import GCNConv, global_mean_pool
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


# Import your custom featurizers
try:
    from canonical_featurizers import featurize_molecule
    print("✅ Custom featurizers imported successfully!")
    print("✅ Ready to proceed to Block 3!")
except ImportError as e:
    print("❌ Error importing custom featurizers:")
    print(e)
    print("Make sure canonical_featurizers.py was uploaded successfully")

In [ ]:
# Block 3: Load DTI Dataset from CAT-DTI
def load_dti_dataset(max_samples=None):
    """Load DTI dataset from CAT-DTI project"""

    print("Loading DTI datasets...")

    try:
        # Change this line to switch datasets
        base_dir = '/home/jovyan/DTI-SL/datasets/human' # Options: human, biosnap, bindingdb
        train_df = pd.read_csv(os.path.join(base_dir, 'train.csv'))
        val_df = pd.read_csv(os.path.join(base_dir, 'val.csv'))
        test_df = pd.read_csv(os.path.join(base_dir, 'test.csv'))

        print(f"✅ Original dataset shapes: Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

        # Concatenate all datasets in the specified order: test, train, val
        combined_df = pd.concat([test_df, train_df, val_df], ignore_index=True)
        print(f"✅ Concatenated Test, Train, and Val shape: {combined_df.shape}")

        # Take a manageable chunk from the combined set if max_samples is specified
        if max_samples and len(combined_df) > max_samples:
            combined_df = combined_df.head(max_samples)
            print(f"Using subset of {max_samples} samples for faster processing")

        print(f"✅ Final combined dataset shape: {combined_df.shape}")
        print(f"Class distribution:\n{combined_df['Y'].value_counts()}")
        print(f"Class balance: {combined_df['Y'].value_counts(normalize=True).round(3)}")

        # Show some statistics for the combined set
        print(f"\n Combined Dataset Statistics:")
        print(f"Total drug-protein pairs: {len(combined_df)}")
        print(f"Average SMILES length: {combined_df['SMILES'].str.len().mean():.1f} characters")
        print(f"Average protein length: {combined_df['Protein'].str.len().mean():.1f} characters")
        print(f"Shortest SMILES: {combined_df['SMILES'].str.len().min()} characters")
        print(f"Longest SMILES: {combined_df['SMILES'].str.len().max()} characters")
        print(f"Shortest protein: {combined_df['Protein'].str.len().min()} characters")
        print(f"Longest protein: {combined_df['Protein'].str.len().max()} characters")

        return combined_df # Return the fully concatenated df

    except Exception as e:
        print(f"❌ Error loading datasets: {e}")
        print("Creating fallback synthetic dataset...")

        # Fallback synthetic dataset
        dti_data = {
            'SMILES': [
                'CCO', 'CC(=O)O', 'c1ccccc1', 'CCN(CC)CC', 'CC(C)O',
                'c1ccc2ccccc2c1', 'CCCCO', 'CC(C)(C)O', 'CC(C)C', 'CCCC'
            ],
            'Protein': [
                'MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG',
                'MKLIVLCSVAVILMGTFMLTFLTQKKAKQRGLL',
                'MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG',
                'MKLIVLCSVAVILMGTFMLTFLTQKKAKQRGLL',
                'MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG',
                'MKLIVLCSVAVILMGTFMLTFLTQKKAKQRGLL',
                'MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG',
                'MKLIVLCSVAVILMGTFMLTFLTQKKAKQRGLL',
                'MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG',
                'MKLIVLCSVAVILMGTFMLTFLTQKKAKQRGLL'
            ],
            'Y': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
        }
        df = pd.DataFrame(dti_data)
        print(f"✅ Fallback dataset created. Shape: {df.shape}")
        return df

# Load DTI dataset
max_samples = None # Remove the max_samples limit for now to process the full combined set
df = load_dti_dataset(max_samples)

print(f"\n✅ Dataset ready!")
print(f"Sample data:")
print(df.head(3))

# Quick test to see the data structure
print(f"\n Sample drug-protein pair:")
print(f"SMILES: {df.iloc[0]['SMILES']}")
print(f"Protein: {df.iloc[0]['Protein'][:100]}...")
print(f"Interaction: {df.iloc[0]['Y']} ({'Yes' if df.iloc[0]['Y'] == 1 else 'No'})")

In [ ]:
# Block 4: Featurize Drugs and Proteins for CAT-DTI
import warnings
warnings.filterwarnings('ignore')

def process_protein_sequence(protein_seq, max_length=1000):
    """
    Process protein sequence similar to dataloader.py
    """
    # Character to integer mapping for amino acids
    CHARPROTSET = {"A": 1, "C": 2, "B": 3, "E": 4, "D": 5, "G": 6,
                   "F": 7, "I": 8, "H": 9, "K": 10, "M": 11, "L": 12,
                   "O": 13, "N": 14, "Q": 15, "P": 16, "S": 17, "R": 18,
                   "U": 19, "T": 20, "W": 21, "V": 22, "Y": 23, "X": 24, "Z": 25}

    def label_sequence(line, smi_ch_ind, MAX_SEQ_LEN=1000):
        X = np.zeros(MAX_SEQ_LEN, np.int64)
        for i, ch in enumerate(line[:MAX_SEQ_LEN]):
            X[i] = smi_ch_ind.get(ch, 0)  # Use 0 for unknown characters
        return X

    # Process protein sequence
    pro_len = len(protein_seq)
    protein_encoded = label_sequence(protein_seq, CHARPROTSET, max_length)

    # Create protein mask
    protein_mask = np.zeros(max_length)
    if pro_len > max_length:
        protein_mask[:] = 1
    else:
        protein_mask[:pro_len] = 1

    return protein_encoded, protein_mask, pro_len

def create_molecular_graph(smiles, max_nodes=290):
    """
    Create PyTorch Geometric graph from SMILES using canonical_featurizers
    """
    try:
        # Use your canonical featurizers
        atom_features, bond_features = featurize_molecule(smiles, self_loop=True)

        # Extract features
        node_features = torch.tensor(atom_features['h'], dtype=torch.float32)
        edge_features = torch.tensor(bond_features['e'], dtype=torch.float32)
        edge_index = torch.tensor(bond_features['edge_indices'], dtype=torch.long)

        # Add virtual node indicator (similar to dataloader.py)
        num_actual_nodes = node_features.shape[0]
        virtual_node_bit = torch.zeros([num_actual_nodes, 1])
        node_features = torch.cat((node_features, virtual_node_bit), 1)

        # Add virtual nodes for padding (similar to dataloader.py)
        num_virtual_nodes = max_nodes - num_actual_nodes
        if num_virtual_nodes > 0:
            virtual_node_feat = torch.cat((
                torch.zeros(num_virtual_nodes, 74),  # 74 atom features
                torch.ones(num_virtual_nodes, 1)     # Virtual node indicator
            ), 1)
            node_features = torch.cat((node_features, virtual_node_feat), 0)

            # Add self-loops for virtual nodes
            virtual_self_loops = torch.stack([
                torch.arange(num_actual_nodes, max_nodes),
                torch.arange(num_actual_nodes, max_nodes)
            ], dim=0)
            edge_index = torch.cat((edge_index, virtual_self_loops), 1)

        # Create PyTorch Geometric Data object
        graph_data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_attr=edge_features,
            num_nodes=max_nodes
        )

        return graph_data, num_actual_nodes

    except Exception as e:
        print(f"Error processing SMILES {smiles}: {e}")
        # Return empty graph as fallback
        empty_features = torch.zeros(max_nodes, 75)  # 74 + 1 virtual indicator
        empty_features[:, -1] = 1  # Mark all as virtual
        empty_edges = torch.stack([torch.arange(max_nodes), torch.arange(max_nodes)], dim=0)
        empty_edge_features = torch.zeros(max_nodes, 13)  # 12 + 1 for self-loops

        return Data(
            x=empty_features,
            edge_index=empty_edges,
            edge_attr=empty_edge_features,
            num_nodes=max_nodes
        ), 0

def process_dti_data(df, max_samples=None, max_nodes=290, max_protein_length=1000):
    """
    Process DTI dataset similar to dataloader.py but for PyTorch Geometric
    """
    print("Processing DTI data...")

    if max_samples and len(df) > max_samples:
        df = df.head(max_samples)
        print(f"Using subset of {max_samples} samples")

    processed_data = []

    for idx, row in df.iterrows():
        try:
            # Process drug (SMILES) as graph
            smiles = row['SMILES']
            drug_graph, num_actual_nodes = create_molecular_graph(smiles, max_nodes)

            # Process protein sequence
            protein_seq = row['Protein']
            protein_encoded, protein_mask, pro_len = process_protein_sequence(
                protein_seq, max_protein_length
            )

            # Convert to tensors
            protein_encoded = torch.tensor(protein_encoded, dtype=torch.long)
            protein_mask = torch.tensor(protein_mask, dtype=torch.long)
            label = torch.tensor(row['Y'], dtype=torch.float32)

            # Store processed data
            sample = {
                'drug_graph': drug_graph,
                'protein_encoded': protein_encoded,
                'protein_mask': protein_mask,
                'protein_length': pro_len,
                'label': label,
                'smiles': smiles,
                'protein_seq': protein_seq,
                'num_actual_nodes': num_actual_nodes
            }

            processed_data.append(sample)

            if (idx + 1) % 100 == 0:
                print(f"Processed {idx + 1}/{len(df)} samples")

        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            continue

    print(f"✅ Successfully processed {len(processed_data)} samples")
    return processed_data

def create_data_loaders(processed_data, batch_size=32, train_ratio=0.8, val_ratio=0.1):
    """
    Create train/val/test data loaders
    """
    print("Creating data loaders...")

    # Split data
    total_samples = len(processed_data)
    train_size = int(total_samples * train_ratio)
    val_size = int(total_samples * val_ratio)
    test_size = total_samples - train_size - val_size

    train_data = processed_data[:train_size]
    val_data = processed_data[train_size:train_size + val_size]
    test_data = processed_data[train_size + val_size:]

    print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

    return train_data, val_data, test_data

def custom_collate_fn(batch):
    """
    Custom collate function for batching graphs and sequences
    """
    # Separate different data types
    drug_graphs = [item['drug_graph'] for item in batch]
    protein_encoded = torch.stack([item['protein_encoded'] for item in batch])
    protein_mask = torch.stack([item['protein_mask'] for item in batch])
    labels = torch.stack([item['label'] for item in batch])

    # Batch the graphs
    batched_graphs = Batch.from_data_list(drug_graphs)

    return {
        'drug_graphs': batched_graphs,
        'protein_encoded': protein_encoded,
        'protein_mask': protein_mask,
        'labels': labels
    }

# Process the dataset
print("🚀 Starting data processing...")
print(f"Dataset shape: {df.shape}")

# Process data
processed_data = process_dti_data(df, max_samples=None, max_nodes=290, max_protein_length=1000)

# Create data loaders
train_data, val_data, test_data = create_data_loaders(
    processed_data, batch_size=32, train_ratio=0.8, val_ratio=0.1
)

print(f"\n✅ Data processing complete!")
print(f"Sample processed data:")
sample = processed_data[0]
print(f"- Drug graph nodes: {sample['drug_graph'].x.shape}")
print(f"- Drug graph edges: {sample['drug_graph'].edge_index.shape}")
print(f"- Protein encoded: {sample['protein_encoded'].shape}")
print(f"- Protein mask: {sample['protein_mask'].shape}")
print(f"- Label: {sample['label']}")
print(f"- SMILES: {sample['smiles'][:50]}...")
print(f"- Protein: {sample['protein_seq'][:50]}...")

# Test batching
print(f"\n🧪 Testing batching...")
batch = custom_collate_fn(train_data[:4])
print(f"Batch drug graphs: {batch['drug_graphs']}")
print(f"Batch protein encoded: {batch['protein_encoded'].shape}")
print(f"Batch labels: {batch['labels'].shape}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_max_pool

# --- Absolute Positional Encoding ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:seq_len, :].unsqueeze(0)

# --- Relative MultiHead Attention (from models.py, simplified for batch_first) ---
class RelativeMultiHeadAttention(nn.Module):
    def __init__(self, d_model=128, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.d_head = d_model // num_heads
        self.num_heads = num_heads
        self.sqrt_dim = d_model ** 0.5

        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)
        self.pos_proj = nn.Linear(d_model, d_model, bias=False)
        self.u_bias = nn.Parameter(torch.Tensor(self.num_heads, self.d_head))
        self.v_bias = nn.Parameter(torch.Tensor(self.num_heads, self.d_head))
        nn.init.xavier_uniform_(self.u_bias)
        nn.init.xavier_uniform_(self.v_bias)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, pos_embedding, mask=None):
        # query, key, value: (batch, seq_len, d_model)
        batch_size, seq_len, _ = query.size()
        # Project and reshape for multi-head
        q = self.query_proj(query).view(batch_size, seq_len, self.num_heads, self.d_head)
        k = self.key_proj(key).view(batch_size, seq_len, self.num_heads, self.d_head)
        v = self.value_proj(value).view(batch_size, seq_len, self.num_heads, self.d_head)
        p = self.pos_proj(pos_embedding).view(batch_size, seq_len, self.num_heads, self.d_head)

        # Transpose for attention: (batch, num_heads, seq_len, d_head)
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)
        p = p.permute(0, 2, 1, 3)

        # Content-based attention
        content_score = torch.matmul((q + self.u_bias.unsqueeze(0).unsqueeze(2)), k.transpose(-2, -1))
        # Position-based attention
        pos_score = torch.matmul((q + self.v_bias.unsqueeze(0).unsqueeze(2)), p.transpose(-2, -1))
        pos_score = self._relative_shift(pos_score)

        score = (content_score + pos_score) / self.sqrt_dim

        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)
            score = score.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(score, -1)
        context = torch.matmul(attn, v)
        context = context.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(context)

    def _relative_shift(self, pos_score):
        # pos_score: (batch, num_heads, seq_len, seq_len)
        batch_size, num_heads, seq_len1, seq_len2 = pos_score.size()
        zeros = pos_score.new_zeros(batch_size, num_heads, seq_len1, 1)
        padded = torch.cat([zeros, pos_score], dim=-1)
        padded = padded.view(batch_size, num_heads, seq_len2 + 1, seq_len1)
        return padded[:, :, 1:].view_as(pos_score)

# --- FeedForward Module (for Transformer) ---
class FeedForwardModule(nn.Module):
    def __init__(self, encoder_dim, expansion_factor=4, dropout_p=0.1):
        super().__init__()
        self.seq = nn.Sequential(
            nn.LayerNorm(encoder_dim),
            nn.Linear(encoder_dim, encoder_dim * expansion_factor),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(encoder_dim * expansion_factor, encoder_dim),
            nn.Dropout(dropout_p)
        )
    def forward(self, x):
        return self.seq(x)

# --- CNN+Transformer Block for Protein ---
class CNNTransBlock(nn.Module):
    def __init__(self, encoder_dim, num_attention_heads, feed_forward_expansion_factor, 
                 feed_forward_dropout_p, attention_dropout_p, conv_dropout_p, conv_kernel_size, max_len):
        super().__init__()
        self.layernorm1 = nn.LayerNorm(encoder_dim)
        self.rel_attn = RelativeMultiHeadAttention(encoder_dim, num_attention_heads)
        self.layernorm2 = nn.LayerNorm(encoder_dim)
        self.conv = nn.Conv1d(encoder_dim, encoder_dim, conv_kernel_size, padding=conv_kernel_size//2)
        self.dropout = nn.Dropout(conv_dropout_p)
        self.ff = FeedForwardModule(encoder_dim, feed_forward_expansion_factor, feed_forward_dropout_p)
        self.layernorm3 = nn.LayerNorm(encoder_dim)
        self.pos_encoder = PositionalEncoding(encoder_dim, max_len)

    def forward(self, x, mask=None):
        # x: (batch, seq_len, encoder_dim)
        pos_emb = self.pos_encoder.pe[:x.size(1), :].unsqueeze(0).expand(x.size(0), -1, -1)
        attn_out = self.rel_attn(x, x, x, pos_emb, mask)
        x = self.layernorm1(x + attn_out)
        # Conv1d expects (batch, channels, seq_len)
        x_conv = x.permute(0, 2, 1)
        x_conv = self.conv(x_conv)
        x_conv = F.relu(x_conv)
        x_conv = self.dropout(x_conv)
        x_conv = x_conv.permute(0, 2, 1)
        x = self.layernorm2(x + x_conv)
        ff_out = self.ff(x)
        x = self.layernorm3(x + ff_out)
        return x

# --- Protein Encoder ---
class ProteinEncoder(nn.Module):
    def __init__(self, vocab_size=26, emb_dim=128, max_len=1000, num_layers=3, num_attention_heads=8, 
                 feed_forward_expansion_factor=4, feed_forward_dropout_p=0.1, 
                 attention_dropout_p=0.1, conv_dropout_p=0.1, conv_kernel_size=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.blocks = nn.ModuleList([
            CNNTransBlock(
                encoder_dim=emb_dim,
                num_attention_heads=num_attention_heads,
                feed_forward_expansion_factor=feed_forward_expansion_factor,
                feed_forward_dropout_p=feed_forward_dropout_p,
                attention_dropout_p=attention_dropout_p,
                conv_dropout_p=conv_dropout_p,
                conv_kernel_size=conv_kernel_size,
                max_len=max_len
            ) for _ in range(num_layers)
        ])
        self.maxpool = nn.AdaptiveMaxPool1d(1)

    def forward(self, seq, mask=None):
        # seq: (batch, seq_len)
        x = self.embedding(seq)  # (batch, seq_len, emb_dim)
        for block in self.blocks:
            x = block(x, mask)
        # Pool over sequence
        x = x.permute(0, 2, 1)  # (batch, emb_dim, seq_len)
        x = self.maxpool(x).squeeze(-1)  # (batch, emb_dim)
        return x

# --- Molecular GCN for Drug Graphs ---
class MolecularGCN(nn.Module):
    def __init__(self, in_feats, dim_embedding=128, hidden_feats=[128,128,128]):
        super().__init__()
        self.init_transform = nn.Linear(in_feats, dim_embedding, bias=False)
        self.gnn_layers = nn.ModuleList()
        last_dim = dim_embedding
        for h in hidden_feats:
            self.gnn_layers.append(GCNConv(last_dim, h))
            last_dim = h
        self.output_feats = last_dim

    def forward(self, x, edge_index, batch):
        x = self.init_transform(x)
        for gnn in self.gnn_layers:
            x = F.relu(gnn(x, edge_index))
        # Pool to get (batch, output_feats)
        x = global_max_pool(x, batch)
        return x

# --- MLP Decoder (as in CATDTI) ---
class MLPDecoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, binary=1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, out_dim)
        self.bn3 = nn.BatchNorm1d(out_dim)
        self.fc4 = nn.Linear(out_dim, binary)
    def forward(self, x):
        x = self.bn1(F.relu(self.fc1(x)))
        x = self.bn2(F.relu(self.fc2(x)))
        x = self.bn3(F.relu(self.fc3(x)))
        x = self.fc4(x)
        return x

# --- Main CATDTI-py Model ---
class CATDTIpy(nn.Module):
    def __init__(self,
                 drug_node_feat_dim=75,
                 drug_embedding=128,
                 drug_hidden_feats=[128,128,128],
                 protein_vocab_size=26,
                 protein_emb_dim=128,
                 protein_max_len=1000,
                 protein_num_layers=3,
                 protein_num_attention_heads=8,
                 protein_ff_expansion=4,
                 protein_ff_dropout=0.1,
                 protein_attn_dropout=0.1,
                 protein_conv_dropout=0.1,
                 protein_conv_kernel=3,
                 mlp_in_dim=256,
                 mlp_hidden_dim=512,
                 mlp_out_dim=128,
                 out_binary=1):
        super().__init__()
        # Drug GNN
        self.drug_extractor = MolecularGCN(
            in_feats=drug_node_feat_dim,
            dim_embedding=drug_embedding,
            hidden_feats=drug_hidden_feats
        )
        # Protein encoder
        self.protein_encoder = ProteinEncoder(
            vocab_size=protein_vocab_size,
            emb_dim=protein_emb_dim,
            max_len=protein_max_len,
            num_layers=protein_num_layers,
            num_attention_heads=protein_num_attention_heads,
            feed_forward_expansion_factor=protein_ff_expansion,
            feed_forward_dropout_p=protein_ff_dropout,
            attention_dropout_p=protein_attn_dropout,
            conv_dropout_p=protein_conv_dropout,
            conv_kernel_size=protein_conv_kernel
        )
        # Multihead Attention (absolute, for cross-modality)
        self.mix_attention_layer = nn.MultiheadAttention(protein_emb_dim, 4, batch_first=True)
        # Dropout and MLP
        self.dropout1 = nn.Dropout(0.1)
        self.mlp_classifier = MLPDecoder(mlp_in_dim, mlp_hidden_dim, mlp_out_dim, binary=out_binary)

    def forward(self, batch, mode="train"):
        # Drug graph
        x, edge_index, batch_idx = batch['drug_graphs'].x, batch['drug_graphs'].edge_index, batch['drug_graphs'].batch
        v_d = self.drug_extractor(x, edge_index, batch_idx)  # (batch, gnn_dim)
        v_d_exp = v_d.unsqueeze(1)  # (batch, 1, gnn_dim) for attention

        # Protein
        v_p = self.protein_encoder(batch['protein_encoded'].long(), batch['protein_mask'].long())  # (batch, emb_dim)
        v_p_exp = v_p.unsqueeze(1)  # (batch, 1, emb_dim) for attention

        # Multihead Attention (drug as query, protein as key/value)
        drug_att, _ = self.mix_attention_layer(v_d_exp, v_p_exp, v_p_exp)
        drug_att = drug_att.squeeze(1)  # (batch, emb_dim)
        # Combine original and attended features
        drug_final = 0.5 * v_d + 0.5 * drug_att
        protein_final = v_p  # (batch, emb_dim)
        # Concatenate and classify
        pair = torch.cat([drug_final, protein_final], dim=1)
        pair = self.dropout1(pair)
        score = self.mlp_classifier(pair)
        return score.squeeze(-1)

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import numpy as np

def move_batch_to_device(batch, device):
    new_batch = {}
    for k, v in batch.items():
        if hasattr(v, 'to'):
            new_batch[k] = v.to(device)
        elif isinstance(v, dict):
            new_batch[k] = move_batch_to_device(v, device)
        else:
            new_batch[k] = v
    return new_batch


# --- Hyperparameters ---
batch_size = 32
num_epochs = 100
learning_rate = 5e-5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- DataLoaders ---
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

# --- Model, Loss, Optimizer ---
model = CATDTIpy().to(device)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# --- Training & Validation Loop ---
def run_epoch(loader, model, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    losses, preds, trues = [], [], []
    for batch in loader:
        # Move tensors to device
        batch = move_batch_to_device(batch, device)
        labels = batch['labels'].float()
        outputs = model(batch)
        loss = criterion(outputs, labels)
        if optimizer:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        losses.append(loss.item())
        preds.append(torch.sigmoid(outputs).detach().cpu().numpy())
        trues.append(labels.detach().cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    return np.mean(losses), preds, trues

for epoch in range(1, num_epochs+1):
    train_loss, train_preds, train_trues = run_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_preds, val_trues = run_epoch(val_loader, model, criterion)
    # Metrics
    train_acc = accuracy_score(train_trues, train_preds > 0.5)
    val_acc = accuracy_score(val_trues, val_preds > 0.5)
    train_auc = roc_auc_score(train_trues, train_preds)
    val_auc = roc_auc_score(val_trues, val_preds)
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f} | "
          f"Train AUC: {train_auc:.3f} | Val AUC: {val_auc:.3f}")

# --- Final Test Evaluation ---
test_loss, test_preds, test_trues = run_epoch(test_loader, model, criterion)
test_acc = accuracy_score(test_trues, test_preds > 0.5)
test_auc = roc_auc_score(test_trues, test_preds)
test_f1 = f1_score(test_trues, test_preds > 0.5)
print(f"\nTest Loss: {test_loss:.4f} | Test Acc: {test_acc:.3f} | Test AUC: {test_auc:.3f} | Test F1: {test_f1:.3f}")